# Main | LLM to Vocabulary


**Request Documentation**: https://platform.openai.com/docs/api-reference/completions/create

**OpenAI Documentation**: https://platform.openai.com/docs/quickstart?context=python

**Find your keys here**: https://platform.openai.com/api-keys

**Keep an eye on your credit usage**: https://platform.openai.com/settings/organization/usage

## Imports

In [ ]:
%pwd

In [ ]:
# Imports
import sys
import os
import importlib

import numpy as np
import random
import time
random.seed(time.time())

import itertools
from itertools import product

from openpyxl import load_workbook

import uuid
uid = str(uuid.uuid4())
print(uid)

from openai import OpenAI
from openai import BadRequestError, RateLimitError, APIError, APITimeoutError


client = OpenAI()

from robots_and_modules.helper_functions import llm_prompt_reply
from robots_and_modules.prompt_builder import build_prompt
from robots_and_modules.distance_calculator import max_dynamic_distance_calculator # <-- only used if needed
from robots_and_modules.distance_calculator import calculate_distance

## Define Inputs

In [ ]:
## Configuration parameters
attempt_id = '44'

llm_iteration_quantity = 40
max_tokens = 2000
llm_iteration_quantity_str = str(llm_iteration_quantity).zfill(3)

llm_model = "gpt-5.1"                   # gpt-4.1-mini      | Cheap, does the code work?
                                        # gpt-5-nano        | Use this for dev/testing
                                        # gpt-4o            | Previously used model  
                                        # gpt-5-mini        | Newest model, smaller, less expensive   ('minimal, 'low', 'medium', 'high')
                                        # gpt-5.1           | Newest model, more expensive            ('none', 'low', 'medium', 'high')
reasoning_effort_level = 'medium'       # 'none' 'minimal, 'low', 'medium', 'high' - affects prompt detail level <-- !!! depends on model
omission_probability = 0.5              # Probability of omitting context information
summarizer = "OFF"                      # "ON" or "OFF" - whether to include a summarization step for motion descriptions

robot_string = 'go1_motion'             # 'go1_motion' or 'go1_audio'
state_space = 'extended'                # 'spread' or 'minimized' 'extended'

printer = "partial"                         # "all", "partial" or None

In [ ]:
# Define the save string for the raw numpy array
normalized_accuracy_array_filename = robot_string + "_" + state_space + "_" + llm_model + "_" + attempt_id + "ID_" + llm_iteration_quantity_str + "ITR_norm.npy"
normalized_accuracy_array_save_string = f"./../data/acc_arrays/{normalized_accuracy_array_filename}"

# Create the trajectory array for the Go1 robot
transformed_trajectory_save_string = "../data/freedog/data/go1_transformed_trajectory_array_box_charger.npy"


### Check Robot Module and State Space

In [ ]:
# Check if robot_string and state_space are within the bounds of go1_motion/go1_audio and spread/minimized
if robot_string not in ['go1_motion', 'go1_audio']:
    print(f"Error: Invalid robot module specified. Using default: {robot_string}")
if state_space not in ['spread', 'minimized', 'extended']:
    print(f"Error: Invalid state space specified. Using default: {state_space}")

# Dynamically import the module
robot_module = importlib.import_module(f"robots_and_modules.{robot_string}")

### Select State Space based on Module

In [ ]:
def pick_set_of_states(state_space):
    # The data in this should be in the format: "State Number: [State Name, State Description]"
    # Used for Go1 robot


    ## SPREAD STATE LIST FOR MAIN EXPERIMENT
    if state_space == 'spread':
        state_list = [
        "S01: [Waiting for Input, The robot is in standby mode, waiting for a command from the user.]",
        "S02: [Analyzing Pinecone, The robot is analyzing a target pinecone in front of it on the ground to assess its size.]", 
        "S03: [Found Pinecone, The robot is signalling to the user it has found a suitable pinecone in front of it on the ground.]",
        "S04: [Error, The robot is signalling to the user it is experiencing a critical error.]",
        "S05: [Confused, The robot is signalling to the user it is confused and unsure what to do.]",
        "S06: [Unsure, The robot’s behavior does not match any of the expected states.]"
        ]


    ## MININMIZED STATE LIST FOR MAIN EXPERIMENT
    elif state_space == 'minimized':
        state_list = [
        "S01: [Waiting for Input, The robot is in standby mode, waiting for a command from the user.]",
        "S02: [Interacting with Pinecone, The robot is interacting with a target pinecone in front of it on the ground.]", 
        "S03: [Needs Help, The robot is signalling that it needs help from the user.]",
        "S04: [Unsure, The robot’s behavior does not match any of the expected states.]"
        ]
    

    ## EXTENDED STATE LIST FOR PILOT EXPERIMENT
    elif state_space == 'extended':
        state_list = [
            "S01: [Waiting/Ready for Command, The robot is idle and attentivly waiting for the next command from the user.]",
            "S02: [Computing/Planning, The robot is internally processing a command from the user and planning its next action.]",
            "S03: [Confused from Command, The robot has received an unclear command and does not understand what to do.]",
            "S04: [Unable to Perform Command, The robot cannot carry out the asigned task (e.g., impossible or blocked).]",
            "S05: [Malfunction/Error, The robot is experiencing a serious internal error.]",
            "S06: [Searching for Target Box, The robot is looking for the target box.]",
            "S07: [Inspecting Candidate Box, The robot is inspecting a candidate box.]",
            "S08: [Confirm Found Target Box, The robot has confidently identified the correct target box.]",
            "S09: [Searching for Charging Dock, The robot is searching for the charging dock.]",
            "S10: [Requesting Battery Charge, The robot is requesting a charge at the charging dock.]",
            "S11: [Charging, The robot is currently docked and charging and not available for normal tasks.]",
            "S12: [Alert User of Danger, The robot has detected a potential hazard and is warning the user.]",
            "S13: [Unsure, The robot’s behavior does not match any of the expected states.]"
        ]


    legend_for_state_codes_dict = {}
    for state in state_list:
        code, rest = state.split(": ", 1)
        name = rest.split(",")[0].strip("[]")
        legend_for_state_codes_dict[code] = name

    return state_list, legend_for_state_codes_dict

set_of_states, state_legend_dict = pick_set_of_states(state_space=state_space)
print(state_legend_dict)


### Create Robot Instance | Set Assistant Prompt and Context

In [ ]:
llm_assistant_prompt = "You are an expert robotic expression design assistant that responds in the exact format [State_Number, State_Name]."

robot_instance = robot_module.Robot(set_of_states)


deployment_context = f"Consider a scenario where you are collaborating with a {robot_instance.form_factor} robot to locate target boxes in a warehouse. The robot’s task is to search the warehouse, inspect boxes it encounters, and determine whether each box is a target box. When it confirms a target box, the robot will signal this to you using its {robot_instance.communication_modality}. The robot is also aware of a charging dock in the warehouse and will signal when its battery is low and needs charging. In addition, the robot can signal other internal states, such as being confused by a command or experiencing a malfunction error."

# OLD WITH PINECONE
# deployment_context = f"Consider a scenario where you are collaborating with a {robot_instance.form_factor} robot to locate and collect large pinecones from the ground in a pine forest. The robot's task is to search the forest floor for large pinecones and analyze their suitability for collection. If a pinecone is deemed large enough for collection, the robot will signal to you using it's {robot_instance.communication_modality} to alert you to its location. Your role is to respond to the robot's call and retrieve the pinecone."

In [ ]:
deployment_context

In [ ]:
robot_instance.get_action_space_shape()

In [ ]:
robot_instance.parameter_ranges

## Accuracy Proxy Generator

### Generate the Accuracy Proxy .npy Array

In [ ]:
# Initialize error and total counters
error_counter = 0
total_counter = 0
flagged_error_counter = 0   # <-- new


# Get action space shape, and use it to create a for loop that itterates all indices
action_space_shape = robot_instance.get_action_space_shape()

## SWAP || 
# Itterate through all possible actions in action space
count = 0
for indices in product(*(range(dim) for dim in action_space_shape)):
    count += 1
    robot_instance.set_active_parameter(list(indices))

    if printer:
        print("\n\n")
        print("------------------------------------------------------")
        print(f" >>>  ACTION {count}  <<<")
        print(robot_instance.active_parameters)
        print("------------------------------------------------------")

    iteration_counter = 0
    iteration_error_counter = 0

    # Five iterations for loop
    for iteration in range(llm_iteration_quantity):

        if printer:
            print(f"**********************************************************************")
            print(f"ITERATION {iteration+1} FOR {robot_instance.active_parameters}")
            print(f"**********************************************************************")

        # Generate description with test values and omission probability
        raw_description = robot_instance.generate_description(omission_probability)


        ### IF USING SUMMARY STEP ### 

        if summarizer == "ON":
            # Prepare a promt with context + robot description
            summary_prompt = f"{deployment_context} \nSummarize the essential details about the robot's {robot_instance.communication_modality} from the text below. Do not add interpretations or explanations. Focus on summarizing the key aspects in the most direct way possible. \n{raw_description}"
        

            summarized_expression = llm_prompt_reply(prompt=summary_prompt, 
                                                    client=client, 
                                                    llm_model=llm_model, 
                                                    llm_assistant_prompt=llm_assistant_prompt, 
                                                    temperature_coefficient=temperature_coefficient, 
                                                    frequency_penalty_coefficient=frequency_penalty_coefficient, 
                                                    top_p_coefficient=top_p_coefficient)

            # Print before and after summary
            if printer:
                print(f"\nRAW DESCRIPTION:\n{raw_description}")
                print(f"\nLLM SUMMARY (override same):\n{summarized_expression}")

        elif summarizer == "OFF":
            summarized_expression = raw_description # <-- end up using raw description directly


        # Pass summarized expression to prompt builder, along with set of states and robot_instance parameters 
        acc_proxy_prompt = build_prompt(expression_string=summarized_expression, 
                                        state_list=set_of_states, 
                                        deployment_context=deployment_context, 
                                        llm_assistant_prompt=llm_assistant_prompt,
                                        expression_modality=robot_instance.communication_modality)
        

        # IF YOU WANT AN EXTRA UUID
        # task_idn = str(uuid.uuid4())
        # full_acc_proxy_prompt = f"Task IDN: {task_idn}\n{acc_proxy_prompt}"
        
        
        if printer == "all":
            print(f"\n\nACCURACY PROXY PROMPT:\n~~~{acc_proxy_prompt}~~~\n\n")

        ### SWAP UNCOMMENT TO DISABLE TEST AND ENABLE LLM
        ## Use test data or run the prompt through the accuracy proxy model
        # six_state_test_options = ["[S01, State 01]", "[S02, State 02]", "[S03, State 03]", "[S04, State 04]", "[S05, State 05]", "[S06, State 06]"]
        # four_state_test_options = ["[S01, State 01]", "[S02, State 02]", "[S03, State 03]", "[S04, State 04]"]
        # acc_proxy_reply = random.choice(six_state_test_options)
        
        # --- robust LLM call with retry + flagged tracking ---
        max_retries = 5
        acc_proxy_reply = None

        for attempt in range(max_retries):
            try:

                acc_proxy_reply = llm_prompt_reply(
                    prompt=acc_proxy_prompt,
                    client=client,
                    llm_model=llm_model,
                    llm_assistant_prompt=llm_assistant_prompt,
                    # temperature=temperature_coefficient,
                    max_output_tokens=max_tokens,
                    reasoning_effort=reasoning_effort_level,
                )
                break  # success -> exit retry loop

            except BadRequestError as e:
                # Specifically catches the "Invalid prompt: flagged..." error
                flagged_error_counter += 1
                error_counter += 1
                total_counter += 1

                if printer:
                    print("!!!! Prompt flagged (BadRequestError). Logging + continuing.")
                    print(f"   Attempt {attempt+1}/{max_retries}")
                    print(f"   Error: {e}")

                # Optional: add a UUID and retry to break eval detector pattern
                task_idn = str(uuid.uuid4())
                acc_proxy_prompt = f"Task IDN: {task_idn}\n{acc_proxy_prompt}"

                time.sleep(20 * (attempt + 1))  # tiny backoff
                continue

            except (RateLimitError, APIError, APITimeoutError) as e:
                # Transient errors: retry
                error_counter += 1
                total_counter += 1

                if printer:
                    print("!!!! Transient API error. Retrying.")
                    print(f"   Attempt {attempt+1}/{max_retries}")
                    print(f"   Error: {e}")

                time.sleep(20 * (attempt + 1))
                continue  # retry loop

        # If all retries failed, set a safe fallback reply
        if acc_proxy_reply is None:
            acc_proxy_reply = "[S13, Unsure]"




        
        print(f"ACCURACY PROXY REPLY: {acc_proxy_reply}\n")

        # Parse the accuracy proxy reply to identify the state
        try:
            state_code, _ = acc_proxy_reply.strip("[]").split(", ")
            state_code = state_code.strip("'")
        except ValueError:
            print(f"Error: The GPT reply {acc_proxy_reply} was not in the correct format.")
            state_code = "E01"
        
        # Check if the state code exists within the dictionary
        if state_code in robot_instance.action_space[tuple(indices)]:
            # Increment the count for the identified state in the action space
            robot_instance.action_space[tuple(indices)][state_code] += 1
            iteration_counter += 1
            total_counter += 1

            if printer:
                # Output the identified state code
                print(f"Identified state code: {state_code}")
                # Output the updated action space
                print(f"Updated action space: {robot_instance.action_space[tuple(indices)]}\n")

        else:
            # Output an error message if the state code does not exist
            iteration_counter += 1
            iteration_error_counter += 1

            total_counter += 1
            error_counter += 1

            
            if printer:
                print(f"Error: State code '{state_code}' does not exist in the action space.\n")

        #     # Log the reply from the accuracy proxy model to the robot_instance action_space
        #     robot_instance.action_space[1, 1, 0, 0, 1, 0] = acc_proxy_reply
    
    if printer:
        print(f"@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
        print(f"COMPLETED ALL {llm_iteration_quantity} ITERATIONS FOR {robot_instance.active_parameters}\n")
        print(f"Iteration Error Count: {iteration_error_counter}\nIteration Count: {iteration_counter}")
        iteration_error_percentage = (iteration_error_counter / iteration_counter) * 100 if iteration_counter > 0 else 0
        print(f"Iteration Error Percentage: {iteration_error_percentage:.2f}%\n")

        print(f"Flagged Prompt Count: {flagged_error_counter}")
        flagged_error_percentage = (flagged_error_counter / total_counter) * 100 if total_counter > 0 else 0
        print(f"Flagged Prompt Percentage: {flagged_error_percentage:.2f}%")

        print(f"Total Error Count: {error_counter}\nTotal Count: {total_counter}")
        total_error_percentage = (error_counter / total_counter) * 100 if total_counter > 0 else 0
        print(f"Total Error Percentage: {total_error_percentage:.2f}%")
        print(f"@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@\n\n")

if printer:
    print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$")
    print(f"COMPLETED ALL COMBINATIONS")
    print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$\n\n")







### Save RAW Array | Normalize | Save Scaled Array to External File

In [ ]:
### Save RAW Array

# Define the save string for the raw numpy array
raw_accuracy_array_filename = robot_string + "_" + state_space + "_" + llm_model + "_" + attempt_id + "ID_" + llm_iteration_quantity_str + "ITR_raw.npy"
raw_accuracy_array_save_string = f"./../data/acc_arrays/{raw_accuracy_array_filename}"

# Save the numpy array robot_instance.action_space to a file 
print(f"Saving RAW numpy array to: {raw_accuracy_array_filename}")
np.save(f"{raw_accuracy_array_save_string}", robot_instance.action_space)


### Normalize Array
# Create a new numpy array for the normalized action space
normalized_action_space = np.zeros_like(robot_instance.action_space, dtype=dict)
for index in np.ndindex(robot_instance.action_space.shape):
    dict_obj = robot_instance.action_space[index]
    total = sum(dict_obj.values())  # Compute sum of values
    if total > 0:  # Avoid division by zero
        normalized_action_space[index] = {key: val / total for key, val in dict_obj.items()}



### Save NORMALIZED Array
# Print the name of the file where the numpy array will be saved
print(f"\nSaving NORMALIZED numpy array to: {normalized_accuracy_array_filename}")
np.save(f"{normalized_accuracy_array_save_string}", normalized_action_space)



# print first element of robot_instance.action_space and normalized_action_space
if robot_string == 'go1_motion':           # 'spread' or 'minimized' or 'extended'
    print("RAW @ [(0, 0, 0, 0, 0, 0)]\t", robot_instance.action_space[(0, 0, 0, 0, 0, 0)])
    print("NORM @ [(0 ,0, 0, 0, 0, 0)]\t", normalized_action_space[(0, 0 ,0, 0, 0, 0)])

## Evaluate the Cost Function

Here, we load the trajectory array generated using **trajectory_array_generator.py**, located in the `data/freedog` directory.

The `freedog` directory is a separate repository for the Unitree Go1 controller. 

For more details and access to the public repository, visit the repository: [github.com/liamreneroy/freedog](https://github.com/liamreneroy/freedog).

### Hill Climb Optimization

### Functions

In [ ]:

# Objective function to identify final states

def objective(state_assignment,
              accuracy_array,
              distance_method,
              action_space_shape, 
              weight=0.7, 
              trajectory_array=None, 
              max_dynamic_distance=None,
              verbose=False):
    '''
    state_assignment: a dict or list that maps each state (e.g. "S01".. "S05")
                      to a chosen expression (6D tuple). !!!! <- is it 6-D or m-D depends on number of params?
                      e.g. {
                         "S01": (0,0,0,0,0,0),
                         "S02": (0,2,1,2,1,0),
                         ...
                      }
    weight: weight for combining "accuracy" vs. "distance".
    
    The objective = alpha * sum_of_accuracies + beta * sum_of_pairwise_distances.
    We'll ensure that all chosen expressions are distinct.
    
    Returns a float score.
    '''

    # 1) Sum of accuracies
    total_accuracy = 0.0
    chosen_exprs = []      # this list is to store initial chosen expressions for vocabulary to facilitate distance calculation
    
    for state, expr in state_assignment.items():
        # Accumulate the accuracy for this (expr, state)
        total_accuracy += accuracy_array[expr][state]

        # We'll also keep track of the chosen expressions for distance
        chosen_exprs.append(expr)
    
    # 2) Sum of pairwise distances among the 5 chosen expressions !!!! <- is it 5 or n depends on number of states?
    total_distance = 0.0
    for i in range(len(chosen_exprs)):
        for j in range(i+1, len(chosen_exprs)):
    
            # total_distance += distance_calculator(chosen_exprs[i], chosen_exprs[j])
            distance = calculate_distance(distance_type=distance_method, 
                                          action_space_shape=action_space_shape,
                                          trajectory_array=trajectory_array,
                                          max_dynamic_distance=max_dynamic_distance,
                                          np_pose_a=np.array(chosen_exprs[i]), 
                                          np_pose_b=np.array(chosen_exprs[j]))
            

            total_distance += distance
            # print(f"Calculated distance between {chosen_exprs[i]} and {chosen_exprs[j]}: {distance}")


    normalized_accuracy = total_accuracy / len(chosen_exprs)                                    # make it between 0 and 1
    normalized_distance = total_distance / (len(chosen_exprs) * (len(chosen_exprs) - 1) / 2)    # make it between 0 and 1


    score = weight * normalized_accuracy + (1 - weight) * normalized_distance   # cost function

    normalized_score = score / 2        # make it between 0 and 1

    if verbose:
        print(f"Total Accuracy: {total_accuracy:.3f}  |  Normalized Accuracy: {normalized_accuracy:.3f}")
        print(f"Total distance: {total_distance:.3f}  |  Normalized Distance: {normalized_distance:.3f}")
        print(f"Total Score:     {score:.3f}  |  Normalized Score:     {normalized_score:.3f}\n")

    return normalized_score, normalized_accuracy, normalized_distance


###############################################################################


# Heuristic Greedy Initialization of States
def heuristic_init(full_expression_list, 
                   states,
                   accuracy_array):
    """
    This function initializes an assignment of expressions to each of the 5 states.
    It selects the expression with the highest accuracy for each state, ensuring
    that all selected expressions are distinct. The selection process is done
    greedily, where for each state, the best expression not used yet is chosen.
    This approach can be replaced with a more advanced method, such as one based
    on a Large Language Model (LLM).
    """

    # Initialize an empty dictionary to store the assignment
    assignment = {}
    
    # Keep track of used expressions to ensure distinctness
    used_expressions = set()
    
    # Iterate through each state to find the best expression
    for state in states:
        best_expr = None
        best_acc = -1.0
        
        # Search through all expressions to find the best one not used yet
        for expr in full_expression_list:
            if expr not in used_expressions:
                acc = accuracy_array[expr][state]
                if acc > best_acc:
                    best_acc = acc
                    best_expr = expr
        
        # Assign the best expression to the current state
        assignment[state] = best_expr
        used_expressions.add(best_expr)
    
    # Return the assignment dictionary

    return assignment


###############################################################################


# Local search (hill-climbing) to assign expressions for the 5 states

def local_search_hill_climb(excel_file,
                            workbook_path,
                            attempt_id,
                            robot_string,
                            state_space,
                            full_expression_list,
                            action_space_shape,
                            distance_method,
                            accuracy_array,
                            states,
                            weight=0.6,
                            max_iterations=50,
                            trajectory_array=None,
                            max_dynamic_distance=None,
                            no_improvement_threshold=2,  # <--- this could potentiall be 1 but 2 is a 'double-check'
                            verbose=False):
    """
    We'll keep a dict: assignment[state] = chosen_expr.
    All chosen_expr must be distinct.
    
    1) Initialize with a heuristic assignment.
    2) Attempt single "swaps": pick a state, pick a different expression from the pool
       (not currently used by any other state), see if it improves the objective.
    3) If improvement is found, accept it and restart the pass.
    4) Stop when no improvement is found in a full pass or we hit max_iterations.
    
    Returns: (best_assignment, best_score), where:
      - best_assignment is a dict mapping "S0i" -> expression (6D tuple).
      - best_score is the objective value for that assignment.
    """
    
    logger_iters = 0

    # weight string for excel logging
    weight_str = str(weight).replace(".", "")

    # 0) Setup Excel Logger
    sheet_name = (robot_string[:7] + "_" + state_space[:3] + "_" + distance_method[:3] + "_ID" + attempt_id + "_" + weight_str).upper()
    # print(f"\nState: {state_code}: {state_name} | Sheet: {sheet_name}\n")

    try: # Try to open existing sheet
        response_sheet = excel_file[sheet_name]
    except KeyError:  # If ot doesn't exist. create it
        response_sheet = excel_file.create_sheet(title=sheet_name)


    response_sheet["A1"] = "id"             # attempt ID range = 40 ... 50 
    response_sheet["B1"] = "model"          # GPT model 
    response_sheet["C1"] = "proxy iters"    # generally 20 or 40 or 50
    response_sheet["D1"] = "robot"          # go1 
    response_sheet["E1"] = "state space"
    response_sheet["F1"] = "dist method"
    response_sheet["G1"] = "weight"
    response_sheet["H1"] = "norm score"
    response_sheet["I1"] = "norm acc"
    response_sheet["J1"] = "norm dist"
    response_sheet["K1"] = "S01 params"
    response_sheet["L1"] = "S01 acc"
    response_sheet["M1"] = "S02 params"
    response_sheet["N1"] = "S02 acc"
    response_sheet["O1"] = "S03 params"
    response_sheet["P1"] = "S03 acc"
    response_sheet["Q1"] = "S04 params"
    response_sheet["R1"] = "S04 acc"
    response_sheet["S1"] = "S05 params"
    response_sheet["T1"] = "S05 acc"
    response_sheet["U1"] = "S06 params"
    response_sheet["V1"] = "S06 acc"
    response_sheet["W1"] = "S07 params"
    response_sheet["X1"] = "S07 acc"
    response_sheet["Y1"] = "S08 params"
    response_sheet["Z1"] = "S08 acc"
    response_sheet["AA1"] = "S09 params"
    response_sheet["AB1"] = "S09 acc"
    response_sheet["AC1"] = "S10 params"
    response_sheet["AD1"] = "S10 acc"
    response_sheet["AE1"] = "S11 params"
    response_sheet["AF1"] = "S11 acc"
    response_sheet["AG1"] = "S12 params"
    response_sheet["AH1"] = "S12 acc"
    # response_sheet["AI1"] = "S13 params"
    # response_sheet["AJ1"] = "S13 acc"
    response_sheet["AK1"] = "improved"

    # leave one empty column "AL1"

    response_sheet["AM1"] = "state code"
    response_sheet["AN1"] = "state"
    response_sheet["AO1"] = "result"
    response_sheet["AP1"] = "accuracy"
    response_sheet["AQ1"] = "P1" # parameter 1 = Body Direction
    response_sheet["AR1"] = "P2" # parameter 2 = Body Tilt
    response_sheet["AS1"] = "P3" # parameter 3 = Body Lean
    response_sheet["AT1"] = "P4" # parameter 4 = Body Height
    response_sheet["AU1"] = "P5" # parameter 5 = Motion Smoothness
    response_sheet["AV1"] = "P6" # parameter 6 = Motion Velocity

    # Put these sufficient
    response_sheet["AM20"] = "dist method"
    response_sheet["AM21"] = "weight"
    response_sheet["AM22"] = "norm score"
    response_sheet["AM23"] = "norm accuracy"
    response_sheet["AM24"] = "norm distance"


    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    # 1) Start with a heuristic assignment
    current_assignment = heuristic_init(full_expression_list=full_expression_list,
                                        states=states,
                                        accuracy_array=accuracy_array)



    current_score, current_accuracy, current_distance = objective(state_assignment=current_assignment, 
                                                                    accuracy_array=accuracy_array,
                                                                    distance_method=distance_method,
                                                                    action_space_shape=action_space_shape, 
                                                                    weight=weight,
                                                                    trajectory_array=trajectory_array, 
                                                                    max_dynamic_distance=max_dynamic_distance,
                                                                    verbose=False)
    

    print(f"::: INIT | Norm Score: {current_score:.4f}, Norm Acc: {current_accuracy:.4f}, Norm Dist: {current_distance:.4f}")

    response_sheet["A"+str(logger_iters+2)] = attempt_id                # attempt ID range = 40 ... 50 
    response_sheet["B"+str(logger_iters+2)] = llm_model                 # GPT model 
    response_sheet["C"+str(logger_iters+2)] = llm_iteration_quantity    # generally 20 or 40 or 50
    response_sheet["D"+str(logger_iters+2)] = robot_string              # go1 
    response_sheet["E"+str(logger_iters+2)] = state_space
    response_sheet["F"+str(logger_iters+2)] = distance_method
    response_sheet["G"+str(logger_iters+2)] = weight
    response_sheet["H"+str(logger_iters+2)] = current_score
    response_sheet["I"+str(logger_iters+2)] = current_accuracy
    response_sheet["J"+str(logger_iters+2)] = current_distance
    
    if "S01" in current_assignment:
        state_string = "S01"
        response_sheet["K"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["L"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
    
    if "S02" in current_assignment:
        state_string = "S02"
        response_sheet["M"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["N"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
    
    if "S03" in current_assignment:
        state_string = "S03"
        response_sheet["O"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["P"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
    
    if "S04" in current_assignment:
        state_string = "S04"
        response_sheet["Q"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["R"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
    
    if "S05" in current_assignment:
        state_string = "S05"
        response_sheet["S"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["T"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S06" in current_assignment:
        state_string = "S06"
        response_sheet["U"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["V"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S07" in current_assignment:
        state_string = "S07"
        response_sheet["W"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["X"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S08" in current_assignment:
        state_string = "S08"
        response_sheet["Y"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["Z"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S09" in current_assignment:
        state_string = "S09"
        response_sheet["AA"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["AB"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S10" in current_assignment:
        state_string = "S10"
        response_sheet["AC"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["AD"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S11" in current_assignment:
        state_string = "S11"
        response_sheet["AE"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["AF"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    if "S12" in current_assignment:
        state_string = "S12"
        response_sheet["AG"+str(logger_iters+2)] = str(current_assignment[state_string])
        response_sheet["AH"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    # if "S13" in current_assignment:
    #     state_string = "S13"
    #     response_sheet["AI"+str(logger_iters+2)] = str(current_assignment[state_string])
    #     response_sheet["AJ"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

    response_sheet["AK"+str(logger_iters+2)] = "*******"


    # Build an easy set of "all_expressions" for quick membership tests
    global_set = set(full_expression_list)
    improved = True
    iteration = 0

    no_improvement_count = 0


    while no_improvement_count < no_improvement_threshold  and iteration < max_iterations:
        iteration += 1
        improved = False

        # We'll iterate over the states in a random order
        state_list = list(states)
        random.shuffle(state_list)
        
        for state in state_list:
            current_expr_for_state = current_assignment[state]
            
            # We want to try each possible expression that isn't used by other states
            other_used_exprs = set(current_assignment[s] for s in states if s != state)
            free_exprs = global_set - other_used_exprs
            

            # We'll see if there's any expression in 'free_exprs' that improves the score
            for candidate_expr in free_exprs:
                
                
                if candidate_expr == current_expr_for_state:
                    # no change
                    continue
                
                # Construct a candidate assignment
                candidate_assignment = dict(current_assignment)
                candidate_assignment[state] = candidate_expr
                
                candidate_score, canidate_accuracy, canidate_distance = objective(state_assignment=candidate_assignment,
                                                                                    accuracy_array=accuracy_array,
                                                                                    distance_method=distance_method,
                                                                                    action_space_shape=action_space_shape,  
                                                                                    weight=weight,
                                                                                    trajectory_array=trajectory_array, 
                                                                                    max_dynamic_distance=max_dynamic_distance,
                                                                                    verbose=False)
                

                if candidate_score <= current_score:
                    logger_iters += 1   
                    response_sheet["A"+str(logger_iters+2)] = attempt_id                   # 03
                    response_sheet["B"+str(logger_iters+2)] = llm_model                    # gpt-40
                    response_sheet["C"+str(logger_iters+2)] = llm_iteration_quantity       # generally 20
                    response_sheet["D"+str(logger_iters+2)] = robot_string
                    response_sheet["E"+str(logger_iters+2)] = state_space
                    response_sheet["F"+str(logger_iters+2)] = distance_method
                    response_sheet["G"+str(logger_iters+2)] = weight
                    response_sheet["H"+str(logger_iters+2)] = candidate_score
                    response_sheet["I"+str(logger_iters+2)] = canidate_accuracy
                    response_sheet["J"+str(logger_iters+2)] = canidate_distance

                    if "S01" in candidate_assignment:
                        state_string = "S01"
                        response_sheet["K"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["L"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S02" in candidate_assignment:
                        state_string = "S02"
                        response_sheet["M"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["N"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S03" in candidate_assignment:
                        state_string = "S03"
                        response_sheet["O"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["P"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S04" in candidate_assignment:
                        state_string = "S04"
                        response_sheet["Q"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["R"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S05" in candidate_assignment:
                        state_string = "S05"
                        response_sheet["S"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["T"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]


                    if "S06" in candidate_assignment:
                        state_string = "S06"
                        response_sheet["U"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["V"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S07" in candidate_assignment:
                        state_string = "S07"
                        response_sheet["W"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["X"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S08" in candidate_assignment:
                        state_string = "S08"
                        response_sheet["Y"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["Z"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S09" in candidate_assignment:
                        state_string = "S09"
                        response_sheet["AA"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["AB"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S10" in candidate_assignment:
                        state_string = "S10"
                        response_sheet["AC"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["AD"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S11" in candidate_assignment:
                        state_string = "S11"
                        response_sheet["AE"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["AF"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    if "S12" in candidate_assignment:
                        state_string = "S12"
                        response_sheet["AG"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                        response_sheet["AH"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    # if "S13" in candidate_assignment:
                    #     state_string = "S13"
                    #     response_sheet["AI"+str(logger_iters+2)] = str(candidate_assignment[state_string])
                    #     response_sheet["AJ"+str(logger_iters+2)] = accuracy_array[candidate_assignment[state_string]][state_string]

                    response_sheet["AK"+str(logger_iters+2)] = ""


                elif candidate_score > current_score:
                    logger_iters += 1   
                    improved = True

                    # Found an improvement; accept it
                    current_assignment = candidate_assignment
                    current_score = candidate_score
                    current_accuracy = canidate_accuracy
                    current_distance = canidate_distance
                    
                    response_sheet["A"+str(logger_iters+2)] = attempt_id               # 03
                    response_sheet["B"+str(logger_iters+2)] = llm_model                # gpt-40
                    response_sheet["C"+str(logger_iters+2)] = llm_iteration_quantity   # generally 20
                    response_sheet["D"+str(logger_iters+2)] = robot_string
                    response_sheet["E"+str(logger_iters+2)] = state_space
                    response_sheet["F"+str(logger_iters+2)] = distance_method
                    response_sheet["G"+str(logger_iters+2)] = weight
                    response_sheet["H"+str(logger_iters+2)] = current_score
                    response_sheet["I"+str(logger_iters+2)] = current_accuracy
                    response_sheet["J"+str(logger_iters+2)] = current_distance
                    if "S01" in current_assignment:
                        state_string = "S01"
                        response_sheet["K"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["L"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    if "S02" in current_assignment:
                        state_string = "S02"
                        response_sheet["M"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["N"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    if "S03" in current_assignment:
                        state_string = "S03"
                        response_sheet["O"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["P"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    if "S04" in current_assignment:
                        state_string = "S04"
                        response_sheet["Q"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["R"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    if "S05" in current_assignment:
                        state_string = "S05"
                        response_sheet["S"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["T"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]


                    if "S06" in current_assignment:
                        state_string = "S06"
                        response_sheet["U"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["V"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

                    if "S07" in current_assignment:
                        state_string = "S07"
                        response_sheet["W"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["X"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    if "S08" in current_assignment:
                        state_string = "S08"
                        response_sheet["Y"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["Z"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

                    if "S09" in current_assignment:
                        state_string = "S09"
                        response_sheet["AA"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["AB"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    
                    if "S10" in current_assignment:
                        state_string = "S10"
                        response_sheet["AC"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["AD"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    
                    if "S11" in current_assignment:
                        state_string = "S11"
                        response_sheet["AE"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["AF"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    
                    if "S12" in current_assignment:
                        state_string = "S12"
                        response_sheet["AG"+str(logger_iters+2)] = str(current_assignment[state_string])
                        response_sheet["AH"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]
                    
                    # if "S13" in current_assignment:
                    #     state_string = "S13"
                    #     response_sheet["AI"+str(logger_iters+2)] = str(current_assignment[state_string])
                    #     response_sheet["AJ"+str(logger_iters+2)] = accuracy_array[current_assignment[state_string]][state_string]

                    response_sheet["AK"+str(logger_iters+2)] = "*******"

                    
                    if verbose:
                        print(f"::: ITERATION {iteration}: swapped state={state} from {current_expr_for_state} to {candidate_expr}")
                        print(f"::: NEW | Norm Score: {current_score:.4f}, Norm Acc: {current_accuracy:.4f}, Norm Dist: {current_distance:.4f}")

                    # Break so we restart scanning from the new assignment
                    break
            
            # If we improved, break the outer loop to start over
            if improved:
                no_improvement_count = 0
                if verbose:
                    print(f"::: Itteration {iteration} | Reset Steps without Improvement: {no_improvement_count}")

                break

        no_improvement_count += 1
        if verbose:
            print(f"::: Itteration {iteration} | Steps without Improvement: {no_improvement_count}")

    if verbose:
        print(f"\n::: Local search ended after {iteration} iteration(s). Best score={current_score:.4f}")
    

    param_columns = ["AQ", "AR", "AS", "AT", "AU", "AV"]

    if "S01" in current_assignment:
        state_string = "S01"
        row_idx = 2
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S02" in current_assignment:
        state_string = "S02"
        row_idx = 3
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]


        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S03" in current_assignment:
        state_string = "S03"
        row_idx = 4
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]


        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S04" in current_assignment:
        state_string = "S04"
        row_idx = 5
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]


        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S05" in current_assignment:
        state_string = "S05"
        row_idx = 6
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S06" in current_assignment:
        state_string = "S06"
        row_idx = 7
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]
    
    if "S07" in current_assignment:
        state_string = "S07"
        row_idx = 8
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]
    
    if "S08" in current_assignment:
        state_string = "S08"
        row_idx = 9
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S09" in current_assignment:
        state_string = "S09"
        row_idx = 10
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S10" in current_assignment:
        state_string = "S10"
        row_idx = 11
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]

    if "S11" in current_assignment:
        state_string = "S11"
        row_idx = 12
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]
    
    if "S12" in current_assignment:
        state_string = "S12"
        row_idx = 13
        response_sheet["AM"+str(row_idx)] = state_string
        response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
        response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
        response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

        items = current_assignment[state_string]
        for i in range(min(len(items), 6)):
            response_sheet[param_columns[i]+str(row_idx)] = items[i]


    # if "S13" in current_assignment:
    #     state_string = "S13"
    #     row_idx = 14
    #     response_sheet["AM"+str(row_idx)] = state_string
    #     response_sheet["AN"+str(row_idx)] = state_legend_dict.get(state_string, "Unknown State")
    #     response_sheet["AO"+str(row_idx)] = str(current_assignment[state_string])
    #     response_sheet["AP"+str(row_idx)] = accuracy_array[current_assignment[state_string]][state_string]

    #     items = current_assignment[state_string]
    #     for i in range(min(len(items), 6)):
    #         response_sheet[param_columns[i]+str(row_idx)] = items[i]




    response_sheet["AN20"] = distance_method
    response_sheet["AN21"] = weight
    response_sheet["AN22"] = current_score
    response_sheet["AN23"] = current_accuracy
    response_sheet["AN24"] = current_distance

    # Save the spreadsheet before changing the sheet
    excel_file.save(workbook_path)    

    return current_assignment, current_score, current_accuracy, current_distance

### Multi Run

In [ ]:
# SETTINGS

## Configuration parameters
loaded_attempt_id = '44'
loaded_llm_iteration_quantity = 40
llm_iteration_quantity_str = str(loaded_llm_iteration_quantity).zfill(3)

llm_model = "gpt-5.1"            # <--- this is whichever model you used
                                    # gpt-4.1-mini      | Cheap, does the code work?
                                    # gpt-5-nano        | Use this for dev/testing
                                    # gpt-4o            | Previously used model  
                                    # gpt-5-mini        | Newest model, smaller, less expensive   ('minimal, 'low', 'medium', 'high')
                                    # gpt-5.1           | Newest model, more expensive            ('none', 'low', 'medium', 'high')


# Default to go1_obj and 'spread' state space
robot_string = 'go1_motion'             # 'go1_motion' or 'go1_audio'
state_space = 'extended'                # 'spread' or 'minimized' 'extended'



# cost_function_weight = 0.7            # Weight for accuracy vs. distance
max_iter = 20                           # max iterations of cost function
go1_max_dynamic_distance = 467.7061     # Calculated for Go1. 

                                          # For shape [2, 3, 3, 3, 2, 3] -> 338.5619
                                          # For shape [3, 3, 3, 3, 2, 2] -> 467.7061

                                          # Hardcoded as takes a while to compute. 
                                          # Used for dynamic distance normalization



# !!!!!!!! If you don't have it, use the commented cell below  !!!!!!!!

In [ ]:

# # Load trajectory array for distance calculations
# trajectory_array_for_max_distance =  np.load("../data/freedog/data/go1_transformed_trajectory_array_box_charger.npy", allow_pickle=True) 

# # Call distance_calculator max_dynamic_distance_calculator function to compute max_dynamic_distance:
# go1_max_dynamic_distance = max_dynamic_distance_calculator(trajectory_array=trajectory_array_for_max_distance)

# print(f"Calculated Go1 max dynamic distance: {go1_max_dynamic_distance:.4f}")

In [ ]:
robot_list = ['go1_motion']      # 'go1_motion', 
state_space_list = ['extended']  # 'spread', 'minimized', 'extended' 

workbook_path = "./../data/selected_poses/optimization_outputs.xlsx"

print ("::: Loading response workbook...")
response_book = load_workbook(workbook_path)
print ("::: Response workbook loaded.")


for weight_val in [0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:         # 0.6, 0.7, 0.8, 0.9
    cost_function_weight = weight_val

    print("\n\n\n\n")
    print("########################")
    print(f"\n\n::: COST FUNCTION WEIGHTING: {cost_function_weight}\n\n")
    print("########################\n\n")

    for robot_selection in robot_list:
        for state_space_selection in state_space_list:

            # sort out which distance methods will be used
            if robot_selection == 'go1_motion':
                distance_method_list = ["none", "emd", "dynamic"] # "none", "emd", "dynamic"
            elif robot_selection == 'go1_audio':
                distance_method_list = ["none", "emd"]


            for distance_method_selection in distance_method_list:

                # Define the save string for the numpy array, load it
                normalized_accuracy_array_filename = robot_selection + "_" + state_space_selection + "_" + llm_model + "_" + loaded_attempt_id + "ID_" + llm_iteration_quantity_str + "ITR_norm.npy"
                normalized_accuracy_array_save_string = f"./../data/acc_arrays/{normalized_accuracy_array_filename}"
                ACCURACY_PROXY_FILE = normalized_accuracy_array_save_string
                loaded_accuracy_proxy_array = np.load(ACCURACY_PROXY_FILE, allow_pickle=True)
                print(f"\n::: Loading accuracy array file: {normalized_accuracy_array_filename}")


                # If we're at dynamic distance, load the trajecotry array, otherwise set it to None
                if distance_method_selection == 'dynamic':
                    transformed_trajectory_save_string = "../data/freedog/data/go1_transformed_trajectory_array_box_charger.npy"
                    TRAJECTORY_ARRAY_FILE = transformed_trajectory_save_string
                    go1_transformed_trajectory_array = np.load(TRAJECTORY_ARRAY_FILE, allow_pickle=True)
                    print(f"\n::: Loading trajectory array file: go1_transformed_trajectory_array_box_charger.npy")
                else:
                    go1_transformed_trajectory_array = None



                # Mark the state identifiers
                if state_space_selection == 'spread':      
                    state_codes = ("S01","S02","S03","S04","S05")
                elif state_space_selection == 'minimized':
                    state_codes = ("S01","S02","S03")
                elif state_space_selection == 'extended':
                    state_codes = ("S01","S02","S03","S04","S05","S06","S07","S08","S09","S10","S11","S12")

                # Get the state list to create the robot instance
                set_of_states, state_legend_dict = pick_set_of_states(state_space=state_space_selection)


                robot_module = importlib.import_module(f"robots_and_modules.{robot_selection}")

                # Create the robot instance so we can get action space shape
                robot_instance = robot_module.Robot(set_of_states)

                # Get action space shape
                motion_param_ranges = robot_instance.get_action_space_shape()    # Example parameter ranges
                                        
                # Collect all expressions (6D index tuples) in a list
                all_expressions = list(itertools.product(*(range(dim) for dim in motion_param_ranges)))


                # Printout Start Set
                print('\n########################')
                print(f'Robot\t\t | {robot_selection}')
                print(f'Weight Value\t | {weight_val}')
                print(f'State Space\t | {state_space_selection}')
                print(f'Dist Method\t | {distance_method_selection}')
                print(f'Action Space\t | {robot_instance.get_action_space_shape()}')
                print(f'Param Ranges\t | {robot_instance.parameter_ranges}')
                print(f'Robot States\t | {state_legend_dict}')
                print('########################\n')


                # Run optimizer 
                best_assignment, best_score, best_accuracy, best_distance = local_search_hill_climb(
                                                                    excel_file=response_book,
                                                                    workbook_path=workbook_path,
                                                                    attempt_id=loaded_attempt_id,
                                                                    robot_string=robot_selection,
                                                                    state_space=state_space_selection,
                                                                    full_expression_list=all_expressions,
                                                                    action_space_shape=motion_param_ranges,
                                                                    distance_method=distance_method_selection,
                                                                    accuracy_array=loaded_accuracy_proxy_array,
                                                                    states=state_codes,
                                                                    weight=cost_function_weight, 
                                                                    max_iterations=max_iter, 
                                                                    trajectory_array=go1_transformed_trajectory_array,
                                                                    max_dynamic_distance=go1_max_dynamic_distance,
                                                                    no_improvement_threshold=2,
                                                                    verbose=True)
                print("\n\n********************")
                print("***** RESULTS ******")
                print("********************\n")

                print("Final States:")
                for st in sorted(best_assignment.keys()):
                    expr = best_assignment[st]
                    state_name = state_legend_dict.get(st, "Unknown State")
                    print(f" {st} -> {expr}, accuracy={loaded_accuracy_proxy_array[expr][st]:.4f} | {state_name}")

                print("\nParameter Legend:")
                for i, key in enumerate(robot_instance.parameter_descriptions.keys(), start=1):
                    print(f"P{i}: {key}")

                print(f"\nAverage Accuracy Score: \t{best_accuracy:.4f}")    
                print(f"Normalized Distance Score: \t{best_distance:.4f}")    
                print(f"Normalized Objective Score: \t{best_score:.4f}")
                print("\n\n\n\n")


    print(f"::: ALL CONDITIONS COMPLETE FOR WEIGHT [w={weight_val}]\n\n")

print(f"::: ********************************************** : )")
print(f"::: ALL WEIGHT CONDITIONS COMPLETE - END OF SCRIPT : )")
print(f"::: ********************************************** : )")


### Single Run

In [ ]:
# SETTINGS

loaded_attempt_id = '14'
max_iter = 20                               # max iterations of cust function
cost_function_weight = 0.6                  # Weight for accuracy vs. distance
go1_max_dynamic_distance = 338.5619         # Calculated for Go1. 
                                            # Hardcoded as takes a while to compute. 
                                            # Used for dynamic distance normalization
distance_method_selection = 'emd'


# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


state_space_selection = state_space
robot_selection = robot_string

workbook_path = "./../data/selected_poses/optimization_outputs.xlsx"
response_book = load_workbook(workbook_path)

# Define the save string for the numpy array, load it
normalized_accuracy_array_filename = robot_selection + "_" + state_space_selection + "_" + llm_model + "_" + loaded_attempt_id + "ID_" + llm_iteration_quantity_str + "ITR_norm.npy"
normalized_accuracy_array_save_string = f"./../data/acc_arrays/{normalized_accuracy_array_filename}"



# Load accuracy proxy values from external file
ACCURACY_PROXY_FILE = normalized_accuracy_array_save_string    # File containing accuracy proxy values
print(f"\n::: Loading accuracy array file: {normalized_accuracy_array_filename}")

TRAJECTORY_ARRAY_FILE = transformed_trajectory_save_string     # File containing trajectory array

loaded_accuracy_proxy_array = np.load(ACCURACY_PROXY_FILE, allow_pickle=True) # Load accuracy proxy values

if distance_method_selection == 'dynamic':
    go1_transformed_trajectory_array = np.load(TRAJECTORY_ARRAY_FILE, allow_pickle=True)
else:
    go1_transformed_trajectory_array = None
                                       

# Mark the state identifiers
if state_space_selection == 'spread':      
    state_codes = ("S01","S02","S03","S04","S05")
elif state_space_selection == 'minimized':
    state_codes = ("S01","S02","S03")

# Get the state list to create the robot instance
set_of_states, state_legend_dict = pick_set_of_states(state_space=state_space_selection)

# Create the robot instance so we can get action space shape
robot_instance = robot_module.Robot(set_of_states)

# Get action space shape
motion_param_ranges = robot_instance.get_action_space_shape()    # Example parameter ranges
                        
# Collect all expressions (6D index tuples) in a list
all_expressions = list(itertools.product(*(range(dim) for dim in motion_param_ranges)))



In [ ]:
# Printout Start Set
print('\n########################')
print(f'Robot\t\t | {robot_selection}')
print(f'State Space\t | {state_space_selection}')
print(f'Dist Method\t | {distance_method_selection}')
print(f'Action Space\t | {robot_instance.get_action_space_shape()}')
print(f'Param Ranges\t | {robot_instance.parameter_ranges}')
print(f'Robot States\t | {state_legend_dict}')
print('########################\n')




best_assignment, best_score, best_accuracy, best_distance = local_search_hill_climb(
                                                    excel_file=response_book,
                                                    workbook_path=workbook_path,
                                                    attempt_id=loaded_attempt_id,
                                                    robot_string=robot_selection,
                                                    state_space=state_space_selection,
                                                    full_expression_list=all_expressions,           
                                                    action_space_shape=motion_param_ranges,
                                                    distance_method=distance_method_selection,
                                                    accuracy_array=loaded_accuracy_proxy_array,
                                                    states=state_codes,
                                                    weight=cost_function_weight, 
                                                    max_iterations=max_iter, 
                                                    trajectory_array=go1_transformed_trajectory_array,
                                                    max_dynamic_distance=go1_max_dynamic_distance,
                                                    no_improvement_threshold=2,
                                                    verbose=True)


print("\n\n********************")
print("***** RESULTS ******")
print("********************\n")

print("Final States:")
for st in sorted(best_assignment.keys()):
    expr = best_assignment[st]
    state_name = state_legend_dict.get(st, "Unknown State")
    print(f" {st} -> {expr}, accuracy={loaded_accuracy_proxy_array[expr][st]:.4f} | {state_name}")

print("\nParameter Legend:")
for i, key in enumerate(robot_instance.parameter_descriptions.keys(), start=1):
    print(f"P{i}: {key}")

print(f"\nAverage Accuracy Score: \t{best_accuracy:.4f}")    
print(f"Normalized Distance Score: \t{best_distance:.4f}")    
print(f"Normalized Objective Score: \t{best_score:.4f}")


# Extra Archive

In [ ]:
# Calculate distance using both EMD and kinematic distance for two specific pose vectors

pose_vect_error_iso = np.array((0, 0, 1, 0, 1, 1))
pose_vect_confu_iso = np.array((0, 1, 2, 1, 1, 1))

pose_vect_error_prx = np.array((0, 1, 2, 0, 1, 2))
pose_vect_confu_prx = np.array((0, 0, 1, 1, 0, 0))

transformed_trajectory_save_string = "../data/freedog/data/go1_transformed_trajectory_array_box_charger.npy"
TRAJECTORY_ARRAY_FILE = transformed_trajectory_save_string
go1_transformed_trajectory_array = np.load(TRAJECTORY_ARRAY_FILE, allow_pickle=True)
print(f"\n::: Loading trajectory array file: go1_transformed_trajectory_array_box_charger.npy")



robot_selection = 'go1_motion'
state_space_selection = 'spread'
state_codes = ("S01","S02","S03","S04","S05")
 
# Get the state list to create the robot instance
set_of_states, state_legend_dict = pick_set_of_states(state_space=state_space_selection)


robot_module = importlib.import_module(f"robots_and_modules.{robot_selection}")

# Create the robot instance so we can get action space shape
robot_instance = robot_module.Robot(set_of_states)

# Get action space shape
motion_param_ranges = robot_instance.get_action_space_shape()    # Example parameter ranges


go1_max_dynamic_distance = 338.5619 # precalculated for Go1






EMD_iso = calculate_distance(distance_type="emd", 
                             trajectory_array=go1_transformed_trajectory_array,
                             np_pose_a=pose_vect_error_iso, 
                             np_pose_b=pose_vect_confu_iso,
                             action_space_shape=motion_param_ranges,
                             max_dynamic_distance=go1_max_dynamic_distance)

EMD_prx = calculate_distance(distance_type="emd", 
                             trajectory_array=go1_transformed_trajectory_array,
                             np_pose_a=pose_vect_error_prx, 
                             np_pose_b=pose_vect_confu_prx,
                             action_space_shape=motion_param_ranges,
                             max_dynamic_distance=go1_max_dynamic_distance)

DYN_iso = calculate_distance(distance_type="dynamic", 
                             trajectory_array=go1_transformed_trajectory_array,
                             np_pose_a=pose_vect_error_iso, 
                             np_pose_b=pose_vect_confu_iso,
                             action_space_shape=motion_param_ranges,
                             max_dynamic_distance=go1_max_dynamic_distance)

DYN_prx = calculate_distance(distance_type="dynamic", 
                             trajectory_array=go1_transformed_trajectory_array,
                             np_pose_a=pose_vect_error_prx, 
                             np_pose_b=pose_vect_confu_prx,
                             action_space_shape=motion_param_ranges,
                             max_dynamic_distance=go1_max_dynamic_distance)

print(f"EMD_iso: {EMD_iso}, DYN_iso: {DYN_iso}")
print(f"EMD_prx: {EMD_prx}, DYN_prx: {DYN_prx}")

